# GWA-T-12 BitBrains
## Source:
https://atlarge-research.com/gwa-t-12/

## Grid description:
The dataset contains the performance metrics of 1,750 VMs from a distributed datacenter from Bitbrains, which is a service provider that specializes in managed hosting and business computation for enterprises. Customers include many major banks (ING), credit card operators (ICS), insurers (Aegon), etc. Bitbrains hosts applications used in the solvency domain; examples of application vendors are Towers Watson and Algorithmics. These applications are typically used for financial reporting, which is used predominately at the end of financial quarters.

Each file contains the performance metrics of a VM. These files are organized according by traces: fastStorage and Rnd.

The first trace, fastStorage, consists of 1,250 VMs that are connected to fast storage area network (SAN) storage devices. The second trace, Rnd, consists of 500 VMs that are either connected to the fast SAN devices or to much slower Network Attached Storage (NAS) devices. The fastStorage trace includes a higher fraction of application servers and compute nodes than the Rnd trace, which is due to the higher performance of the storage attached to the fastStorage machines. Conversely, for the Rnd trace we observe a higher fraction of management machines, which only require storage with lower performance and less frequent access.

In the Rnd directory, the files are organized into 3 sub-directories by the month that the metrics are recorded.

The format of each file is row-based, each row represent an observation of the performance metrics. Each column of a row is separate by “;\t” The format of each row is

Timestamp: number of milliseconds since 1970-01-01.
CPU cores: number of virtual CPU cores provisioned.
CPU capacity provisioned (CPU requested): the capacity of the CPUs in terms of MHZ, it equals to number of cores x speed per core.
CPU usage: in terms of MHZ.
CPU usage: in terms of percentage
Memory provisioned (memory requested): the capacity of the memory of the VM in terms of KB.
Memory usage: the memory that is actively used in terms of KB.
Disk read throughput: in terms of KB/s
Disk write throughput: in terms of KB/s
Network received throughput: in terms of KB/s
Network transmitted throughput: in terms of KB/s

## Fast Storage Trace:
The fastStorage trace contains the performance metrics of 1,250 VMs. Each VM is processed as an OpenDC task.

## OpenDC Tasks and Fragments documentation
source: https://atlarge-research.github.io/opendc/docs/documentation/Input/Workload
```
Workloads define what tasks in the simulation, when they were submitted, and their computational requirements. Workload are defined using two files:

Tasks: The Tasks file contains the metadata of the tasks
Fragments: The Fragments file contains the computational demand of each task over time
Both files are provided using the parquet format.

Tasks
The Tasks file provides an overview of the tasks:

Metric	Required?	Datatype	Unit	Summary
id	Yes	string		The id of the server
submission_time	Yes	int64	datetime	The submission time of the server
nature	No	string	[deferrable, non-deferrable]	Defines if a task can be delayed
deadline	No	string	datetime	The latest the scheduling of a task can be delayed to.
duration	Yes	int64	datetime	The finish time of the submission
cpu_count	Yes	int32	count	The number of CPUs required to run this task
cpu_capacity	Yes	float64	MHz	The amount of CPU required to run this task
mem_capacity	Yes	int64	MB	The amount of memory required to run this task
gpu_count	No	int32	count	The number of GPUs required to run this task
gpu_capacity	No	float64	MHz	The amount of GPU required to run this task
gpu_mem_capacity	No	int64	MB	The amount of memory required to run this task. (Currently ignored)
Fragments
The Fragments file provides information about the computational demand of each task over time:

Metric	Required?	Datatype	Unit	Summary
id	Yes	string		The id of the task
duration	Yes	int64	milli seconds	The duration since the last sample
cpu_count	Yes	int32	count	The number of cpus required
cpu_usage	Yes	float64	MHz	The amount of computational CPU power required.
gpu_count	No	int32	count	The number of gpus required
gpu_usage	No	float64	MHz	The amount of computational GPU power required.
```

In [14]:
import pandas as pd
import numpy as np
import os

In [15]:
folder_location = "../raw_traces/raw_fastStorage_2013-8"
vms = [pd.read_csv(folder_location + "/" + file, sep=";\t", header=None) for file in os.listdir(folder_location)] # each file is a VM, and each of these VMs will be outputted as an OpenDC task -> 1250 tasks

/var/folders/zp/wbw59jc53p912jytp6zlm1wr0000gs/T/ipykernel_26041/1839167307.py:2: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  vms = [pd.read_csv(folder_location + "/" + file, sep=";\t", header=None) for file in os.listdir(folder_location)] # each file is a VM, and each of these VMs will be outputted as an OpenDC task -> 1250 tasks
/var/folders/zp/wbw59jc53p912jytp6zlm1wr0000gs/T/ipykernel_26041/1839167307.py:2: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  vms = [pd.read_csv(folder_location + "/" + file, sep=";\t", header=None) for file in os.listdir(folder_location)] # each file is a VM, and each of 

In [16]:
# convert each dataframe to a task, and then concatenate all tasks into a single dataframe
tasks = []
fragments = []

for vm in vms:
    task = {}
    task["id"] = vm.iloc[0, 0] # use the timestamp of the first observation as the id of the task
    task["submission_time"] = vm.iloc[0, 0] # use the timestamp of the first observation as the submission time of the task
    task["nature"] = "deferrable" # we can delay the scheduling of the task, as we do not have information about
    task["deadline"] = vm.iloc[-1, 0] # use the timestamp of the last observation as the deadline of the task
    task["duration"] = vm.iloc[-1, 0] - vm.iloc[0, 0] # use the difference between the timestamp of the last observation and the timestamp of the first observation as the duration of the task
    task["cpu_count"] = vm.iloc[0, 1] # use the number of CPU cores provisioned in the first observation as the number of CPU cores required to run the task
    task["cpu_capacity"] = vm.iloc[0, 2] # use the CPU capacity provisioned in the first observation as the amount of CPU required to run the task
    task["mem_capacity"] = vm.iloc[0, 5] / 1024 # use the memory provisioned in the first observation as the amount of memory required to run the task, convert from KB to MB
    tasks.append(task)

    for i in range(len(vm)):
        fragment = {}
        fragment["id"] = task["id"] # use the id of the task as the id of the fragment
        fragment["duration"] = vm.iloc[i, 0] - vm.iloc[i-1, 0] if i > 0 else 0 # use the difference between the timestamp of the current observation and the timestamp of the previous observation as the duration of the fragment, for the first observation, we can set the duration to 0
        fragment["cpu_count"] = vm.iloc[i, 1] # use the number of CPU cores provisioned in the current observation as the number of CPU cores required to run the fragment
        fragment["cpu_usage"] = vm.iloc[i, 3] # use the CPU usage in terms of percentage in the current observation as the amount of computational CPU power required to run the fragment
        fragments.append(fragment)

TypeError: unsupported operand type(s) for -: 'str' and 'str'